In [1]:
import torch
import torch.nn as nn

In [2]:
class MLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer_1 = nn.Linear(6,4, bias=False)
    self.layer_2 = nn.Linear(4,2, bias=False)
    self.layer_3 = nn.Linear(2,2, bias=False)

  def forward(self, x):
    x = self.layer_1(x)
    x = self.layer_2(x)
    x = self.layer_3(x)

    return x

In [3]:
inp = torch.tensor([1,2,3,4,5,6], dtype=torch.float32)
targets = torch.tensor(0)

In [4]:
torch.manual_seed(123)

mlp = MLP()

In [5]:
logits = mlp(inp)
print(logits)
loss = nn.functional.cross_entropy(logits, targets)
print(loss)

tensor([0.7440, 0.9682], grad_fn=<SqueezeBackward4>)
tensor(0.8115, grad_fn=<NllLossBackward0>)


In [6]:
loss.backward()
mlp.layer_1.weight.grad

tensor([[-0.1204, -0.2408, -0.3612, -0.4816, -0.6020, -0.7223],
        [ 0.1231,  0.2463,  0.3694,  0.4925,  0.6157,  0.7388],
        [-0.1258, -0.2516, -0.3774, -0.5032, -0.6290, -0.7548],
        [ 0.0045,  0.0091,  0.0136,  0.0181,  0.0226,  0.0272]])

In [7]:
mlp.layer_2.weight.grad

tensor([[-0.0117, -0.1271, -0.2434,  0.6620],
        [ 0.0161,  0.1748,  0.3346, -0.9101]])

In [8]:
mlp.layer_3.weight.grad

tensor([[-0.9769, -0.3865],
        [ 0.9769,  0.3865]])

In [9]:
mlp.zero_grad()

In [10]:
#another approach first mat mul the weights of all the layers and then pass the input
mat_mul_output = (mlp.layer_1.weight.T @ mlp.layer_2.weight.T) @ mlp.layer_3.weight.T
mat_mul_output.shape

torch.Size([6, 2])

In [11]:
logits_2 = inp @ mat_mul_output
logits_2

tensor([0.7440, 0.9682], grad_fn=<SqueezeBackward4>)

In [12]:
loss_2 = nn.functional.cross_entropy(logits_2, targets)
loss_2

tensor(0.8115, grad_fn=<NllLossBackward0>)

In [13]:
loss_2.backward()

In [14]:
mlp.layer_1.weight.grad

tensor([[-0.1204, -0.2408, -0.3612, -0.4816, -0.6020, -0.7223],
        [ 0.1231,  0.2463,  0.3694,  0.4925,  0.6157,  0.7388],
        [-0.1258, -0.2516, -0.3774, -0.5032, -0.6290, -0.7548],
        [ 0.0045,  0.0091,  0.0136,  0.0181,  0.0226,  0.0272]])

In [15]:
mlp.layer_2.weight.grad

tensor([[-0.0117, -0.1271, -0.2434,  0.6620],
        [ 0.0161,  0.1748,  0.3346, -0.9101]])

In [16]:
mlp.layer_3.weight.grad

tensor([[-0.9769, -0.3865],
        [ 0.9769,  0.3865]])

- both the approaches yield same result in terms of gradients, logits and loss